In [0]:
%sql
-- Calculate running total of revenue per region.
WITH daily_revenue AS (SELECT region_code, DATE(order_date) AS order_date, ROUND( SUM(oi.quantity * oi.unit_price *(1 - oi.discount_percent / 100)),2)
AS daily_revenue FROM orders o JOIN order_items oi ON o.order_id = oi.order_id WHERE o.customer_id IS NOT NULL GROUP BY region_code, DATE(order_date))
SELECT region_code, order_date, daily_revenue, ROUND( SUM(daily_revenue) OVER ( PARTITION BY region_code ORDER BY order_date 
ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2) AS running_total FROM daily_revenue ORDER BY region_code, order_date;

region_code,order_date,daily_revenue,running_total
109909,2024-08-10,27970.76,27970.76
109909,2024-09-29,28051.98,56022.74
109909,2024-09-30,11741.6,67764.34
109909,2024-10-08,21795.45,89559.79
109909,2024-10-22,127883.21,217443.0
109909,2024-10-27,276010.98,493453.98
109909,2024-11-12,54624.45,548078.43
109909,2024-11-27,1239.39,549317.82
109909,2024-12-19,50976.74,600294.56
109909,2024-12-23,336571.09,936865.65


In [0]:
%sql
-- Ranking Products by Revenue within Category
WITH product_revenue AS (SELECT p.category, p.product_name, ROUND( SUM( oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100)),2)
AS total_revenue FROM order_items oi JOIN products p ON oi.product_id = p.product_id GROUP BY p.category, p.product_name)
SELECT category, product_name, total_revenue, DENSE_RANK() OVER ( PARTITION BY category ORDER BY total_revenue DESC) 
AS rank_in_category FROM product_revenue ORDER BY category, rank_in_category;

category,product_name,total_revenue,rank_in_category
Beauty,Lipstick,4370974.29,1
Beauty,Perfume,3006267.94,2
Beauty,Body Lotion,2717095.15,3
Beauty,Shampoo,1862668.01,4
Beauty,Face Wash,1734292.69,5
Books,Ai Basics,634931.7,1
Books,Machine Learning,523082.45,2
Books,Sql Handbook,488374.76,3
Books,Java Programming,472867.69,4
Books,Python Guide,281284.71,5


In [0]:
%sql
-- LAG/LEAD Analysis
WITH customer_orders AS ( SELECT customer_id, DATE(order_date) AS order_date, LAG(DATE(order_date)) OVER ( PARTITION BY customer_id ORDER BY DATE(order_date)) 
AS previous_order_date FROM orders WHERE customer_id IS NOT NULL), order_gaps AS (SELECT customer_id, order_date, previous_order_date, DATEDIFF( order_date, 
previous_order_date) AS days_gap FROM customer_orders)SELECT customer_id, order_date, previous_order_date, days_gap, CASE WHEN AVG(days_gap) 
OVER ( PARTITION BY customer_id ) > 30 THEN 'At Risk' ELSE 'Active'END AS customer_status FROM order_gaps ORDER BY customer_id, order_date;

customer_id,order_date,previous_order_date,days_gap,customer_status
1,2025-01-19,null,null,At Risk
1,2025-02-28,2025-01-19,40,At Risk
1,2025-03-11,2025-02-28,11,At Risk
1,2025-06-22,2025-03-11,103,At Risk
1,2025-08-01,2025-06-22,40,At Risk
1,2025-08-23,2025-08-01,22,At Risk
1,2025-10-24,2025-08-23,62,At Risk
1,2026-03-08,2025-10-24,135,At Risk
1,2026-07-28,2026-03-08,142,At Risk
1,2026-08-03,2026-07-28,6,At Risk


In [0]:
%sql
-- CTE with Multiple Levels
WITH monthly_customer_revenue AS ( SELECT o.customer_id, DATE_FORMAT(o.order_date, 'yyyy-MM') AS order_month,
ROUND(SUM( oi.quantity * oi.unit_price *(1 - oi.discount_percent / 100)),2) AS monthly_revenue FROM orders o JOIN order_items oi ON o.order_id = oi.order_id 
WHERE o.customer_id IS NOT NULL GROUP BY o.customer_id, DATE_FORMAT(o.order_date, 'yyyy-MM')), customer_categories AS ( SELECT customer_id, order_month, monthly_revenue, CASE
WHEN monthly_revenue > 10000 THEN 'High'WHEN monthly_revenue >= 5000 THEN 'Medium' ELSE 'Low' END AS revenue_category FROM monthly_customer_revenue) 
SELECT order_month, revenue_category, COUNT(*) AS customer_count FROM customer_categories GROUP BY order_month, revenue_category ORDER BY order_month, revenue_category;

order_month,revenue_category,customer_count
2024-08,High,98
2024-08,Low,20
2024-08,Medium,12
2024-09,High,113
2024-09,Low,31
2024-09,Medium,17
2024-10,High,149
2024-10,Low,40
2024-10,Medium,18
2024-11,High,146


In [0]:
%sql
--  NTILE for Segmentation
WITH customer_lifetime_value AS (SELECT o.customer_id, ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100)),2) AS total_value
FROM orders o JOIN order_items oi ON o.order_id = oi.order_id WHERE o.customer_id IS NOT NULL GROUP BY o.customer_id),customer_quartiles 
AS (SELECT customer_id,total_value,NTILE(4) OVER (ORDER BY total_value DESC) AS quartile FROM customer_lifetime_value)SELECT customer_id,total_value,quartile, 
CASE WHEN quartile = 1 THEN 'Platinum' WHEN quartile = 2 THEN 'Gold' WHEN quartile = 3 THEN 'Silver' WHEN quartile = 4 THEN 'Bronze' END AS quartile_label
FROM customer_quartiles ORDER BY quartile,total_value DESC;

customer_id,total_value,quartile,quartile_label
248,1256409.24,1,Platinum
15,1211641.0,1,Platinum
695,1189463.41,1,Platinum
771,1080227.64,1,Platinum
372,1057590.69,1,Platinum
501,1034852.92,1,Platinum
315,1024323.44,1,Platinum
450,993905.91,1,Platinum
364,970581.78,1,Platinum
561,940514.75,1,Platinum


In [0]:
%sql
--  Year-over-Year Comparison
WITH monthly_revenue AS (SELECT DATE_TRUNC('month',o.order_date) AS month_date, ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100)),2) AS revenue
FROM orders o JOIN order_items oi ON o.order_id = oi.order_id GROUP BY DATE_TRUNC('month',o.order_date)) SELECT YEAR(m.month_date) AS year,MONTH(m.month_date) 
AS month,m.revenue,p.revenue AS prev_year_revenue, CASE WHEN p.revenue IS NULL OR p.revenue = 0 THEN NULL ELSE ROUND(((m.revenue - p.revenue) / p.revenue) * 100,2) END AS yoy_growth_percent
FROM monthly_revenue m LEFT JOIN monthly_revenue p ON p.month_date = ADD_MONTHS(m.month_date,-12) ORDER BY year,month;

year,month,revenue,prev_year_revenue,yoy_growth_percent
2024,8,7250171.7,null,null
2024,9,9229664.38,null,null
2024,10,1.183251364E7,null,null
2024,11,1.229009357E7,null,null
2024,12,1.144204894E7,null,null
2025,1,8184275.71,null,null
2025,2,1.354475369E7,null,null
2025,3,1.072863623E7,null,null
2025,4,1.106371395E7,null,null
2025,5,1.317958265E7,null,null


In [0]:
%sql

--  First/Last Value Analysis
WITH customer_category_orders AS (SELECT o.customer_id,o.order_date,p.category, FIRST_VALUE(p.category) OVER (PARTITION BY o.customer_id ORDER BY o.order_date) AS first_category,
LAST_VALUE(p.category) OVER (PARTITION BY o.customer_id ORDER BY o.order_date ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS latest_category
FROM orders o JOIN order_items oi ON o.order_id = oi.order_id JOIN products p ON oi.product_id = p.product_id WHERE o.customer_id IS NOT NULL)
SELECT DISTINCT customer_id,first_category,latest_category, CASE WHEN first_category = latest_category THEN 'No' ELSE 'Yes' END AS category_shift
FROM customer_category_orders ORDER BY customer_id;

customer_id,first_category,latest_category,category_shift
1,Beauty,Electronics,Yes
2,Clothing,Electronics,Yes
3,Electronics,Books,Yes
4,Electronics,Home,Yes
5,Books,Books,No
6,Sports,Clothing,Yes
7,Sports,Sports,No
8,Beauty,Beauty,No
9,Beauty,Home,Yes
10,Beauty,Sports,Yes


In [0]:
%sql
--  Cumulative Distribution
WITH customer_revenue AS (SELECT o.customer_id, ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100)),2) AS revenue
FROM orders o JOIN order_items oi ON o.order_id = oi.order_id WHERE o.customer_id IS NOT NULL GROUP BY o.customer_id), cumulative AS (SELECT customer_id,revenue,
SUM(revenue) OVER (ORDER BY revenue DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_revenue, SUM(revenue) OVER () AS total_revenue FROM customer_revenue) 
SELECT customer_id,revenue,cumulative_revenue, ROUND((cumulative_revenue / total_revenue) * 100,2) AS cumulative_percent FROM cumulative ORDER BY revenue DESC;

customer_id,revenue,cumulative_revenue,cumulative_percent
248,1256409.24,1256409.24,0.48
15,1211641.0,2468050.24,0.94
695,1189463.41,3657513.6500000004,1.39
771,1080227.64,4737741.29,1.8
372,1057590.69,5795331.98,2.2
501,1034852.92,6830184.9,2.59
315,1024323.44,7854508.34,2.98
450,993905.91,8848414.25,3.35
364,970581.78,9818996.03,3.72
561,940514.75,1.075951078E7,4.08


In [0]:
%sql
--  Self-Join with Window Function
WITH product_pairs AS (SELECT oi1.product_id AS product_a_id, oi2.product_id AS product_b_id, COUNT(DISTINCT oi1.order_id) AS times_bought_together
FROM order_items oi1 JOIN order_items oi2 ON oi1.order_id = oi2.order_id AND oi1.product_id < oi2.product_id GROUP BY oi1.product_id,oi2.product_id)
SELECT p1.product_name AS product_a, p2.product_name AS product_b, pp.times_bought_together FROM product_pairs pp
JOIN products p1 ON pp.product_a_id = p1.product_id JOIN products p2 ON pp.product_b_id = p2.product_id ORDER BY pp.times_bought_together DESC;

product_a,product_b,times_bought_together
Perfume,Microwave,6
Lipstick,Lipstick,6
Laptop,Shampoo,6
Body Lotion,Tablet,6
T-Shirt,Sweater,6
Python Guide,Machine Learning,6
Microwave,Speaker,6
Java Programming,Lipstick,6
Dumbbells,Speaker,6
Machine Learning,Yoga Mat,6
